In [ ]:
import pandas as pd

df = pd.read_csv("facturas.csv")
df.head()

## Ejercicio 1
Comprobar que las facturas se han cargado correctamente.

In [ ]:
# 1. Número de filas y columnas
print("Filas y columnas:", df.shape)

# 2. Tipo de dato de cada columna
print("\nTipos de datos:")
print(df.dtypes)

# 3. Valores vacíos por columna
print("\nValores vacíos:")
print(df.isna().sum())

# 4. Importe total de las facturas
print("\nImporte total:", df["importe"].sum())

In [ ]:
# La fecha esta en formato str por lo que lo pasamos a formato fecha con datetime
df["fecha"] = pd.to_datetime(df ["fecha"])

## Ejercicio 2
Detectar posibles duplicados.

In [ ]:
columnas_clave = ["proveedor", "factura", "importe"]

mascara_duplicados = df.duplicated(
    subset=columnas_clave,
    keep=False
)

df_duplicados = df[mascara_duplicados]

df_duplicados

Estas lineas podrían tratarse de una factura que se ha registrado por duplicado, pero hay que ver que no se trate de una partición de la misma factura porque tienen 2 vencimientos distintos.

# 2.1. Tratamiento con duplicados

In [ ]:
# Crear una tabla de control sin modificar el original

df_control = df.copy()

columnas_clave = ["proveedor", "factura", "importe"]

coincidencias = df_control.duplicated(
    subset=columnas_clave,
    keep=False
)

df_control["estado_revision"] = "Sin alerta por coincidencia"
df_control["motivo_alerta"] = ""

df_control.loc[coincidencias, "estado_revision"] = "Pendiente de validar"
df_control.loc[coincidencias, "motivo_alerta"] = (
    "Coinciden proveedor, factura e importe."
    "Comprobar documento contable, posición y vencimiento."
)

In [ ]:
# Mostrar las partidas que necesitan investigación

df_revision = df_control.loc[coincidencias].copy()

df_revision

In [ ]:
# Exportar la lista para revisarla con el ERP

df_revision.to_csv(
    "facturas_pendientes_revision.csv",
    index=False,
    encoding="utf-8-sig"
)

## Ejercicio 3
Detectar importes inusuales que necesitan revisión. 

In [ ]:
# Calcular la referencia sobre todas las partidas
mediana = df_control["importe"].median()
umbral = mediana * 3

print("Mediana:", mediana)
print("Umbral de revisión:", umbral)

# Añadir una alerta independiente de la coincidencia
df_control["alerta_importe"] = df_control["importe"] > umbral

# Mostrar las partidas que superan el umbral
df_importes_altos = df_control.loc[
    df_control["alerta_importe"],
    ["registro", "proveedor", "factura", "importe", "alerta_importe"]
].copy()

df_importes_altos

**Conclusión:** La factura F205, de 7.800€, supera el umbral orientativo de 2.850€. Antes de incluirla en la remesa, comprobaría su justificación y aprobación, los anticipos o pagos previos, el importe pendiente y el vencimiento. Después contrastaría el pago con el saldo disponible y los demás compromisos de tesorería. La alerta no demuestra un error.

## Ejercicio 4
Generar un informe único de alertas para Tesorería.

In [ ]:
# Recuperar la alerta por coincidencia
df_control["alerta_coincidencia"] = df_control.duplicated(
    subset=["proveedor", "factura", "importe"],
    keep=False
)

# Seleccionar partidas con al menos una alerta
hay_alerta = (
    df_control["alerta_coincidencia"]
    | df_control["alerta_importe"]
)

df_alertas = df_control.loc[hay_alerta].copy()

# Construir el motivo, conservando ambas alertas si coinciden
df_alertas["motivo_alerta"] = ""

df_alertas.loc[
    df_alertas["alerta_coincidencia"], "motivo_alerta"
] = "Coincidencia de proveedor, factura e importe. "

df_alertas.loc[
    df_alertas["alerta_importe"], "motivo_alerta"
] += "Importe superior a tres veces la mediana."

df_alertas[
    ["registro", "importe", "alerta_importe", "motivo_alerta"]
]

df_alertas["estado_revision"] = "Pendiente de validar"

# Elegir las columnas del informe
columnas_informe = [
    "registro", "proveedor", "factura", "fecha", "importe",
    "alerta_coincidencia", "alerta_importe",
    "motivo_alerta", "estado_revision"
]

df_alertas = df_alertas[columnas_informe]

df_alertas

In [ ]:
# Exportación
df_alertas.to_csv(
    "alertas_tesoreria.csv",
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

print("Partidas para revisar:", len(df_alertas))
print("Importe de las partidas señaladas:", df_alertas["importe"].sum())

## Ejercicio 5 — Validación del informe

In [ ]:
informe = pd.read_csv("alertas_tesoreria.csv", sep=";", encoding="utf-8-sig")
assert len(informe) == 3, "Número de partidas incorrecto"
assert set(informe["registro"]) == {1, 3, 6}, "Registros incorrectos"
assert informe["importe"].sum() == 10200, "Importe total incorrecto"
texto_importe = informe["motivo_alerta"].str.contains("Importe superior", regex=False, na=False)
assert texto_importe.equals(informe["alerta_importe"]), "El motivo no coincide con la alerta"
print("Validación superada: el informe exportado es coherente.")